The ``Statistics`` standard library module provides functions to compute the mean and deviation of the wait times.

In [1]:
using Statistics

For the arrival times, we will use the Poisson distribution, provided in the package StatsKit, which needs to be installed separately, as it is not part of the standard library.

In [2]:
using StatsKit

To format the numbers when printing arrival times, we only show two decimals.

In [3]:
using Printf
Base.show(io::IO, f::Float64) = @printf(io, "%.2f", f)

# The Monte Carlo method

The Monte Carlo method is illustrated on three problems.

# 1. The Newsboy Problem

In [4]:
"""
A newspaper seller has on average 100 customers a day
as 5000 people are passing by the shop each day.

The seller has to buy each paper at 50 cents a copy,
sells it at 75 cents, with no returns. 
For each sold paper, the seller makes a profit of 25 cents,
the loss is 75 cents for each unsold paper.

To maximize profit, how many copies should the seller buy?
"""
function newsboy()
    (maxprofit, maxidx) = (0, 0)
    for nbpapers = 90:110
        sumday = 0
        for day=1:365
            profit = -nbpapers*0.50
            nbsold = 0
            for passerby=1:5000
                if nbsold < nbpapers
                    if rand() <= 1.0/50
                        profit = profit + 0.75
                        nbsold = nbsold + 1
                    end
                end
            end
            sumday = sumday + profit
        end
        avgday = sumday/365
        println("profit from ", nbpapers, " : \$", avgday)
        if maxprofit < avgday
            maxprofit = avgday
            maxidx = nbpapers
        end
    end
    print(" maximum profit : \$", maxprofit)
    println(" at ", maxidx)
end

newsboy

In [5]:
newsboy()

profit from 90 : $21.96
profit from 91 : $22.14
profit from 92 : $22.35
profit from 93 : $22.45
profit from 94 : $22.28
profit from 95 : $22.18
profit from 96 : $22.06
profit from 97 : $22.36
profit from 98 : $22.49
profit from 99 : $22.54
profit from 100 : $22.07
profit from 101 : $21.88
profit from 102 : $21.24
profit from 103 : $21.73
profit from 104 : $21.24
profit from 105 : $21.16
profit from 106 : $20.81
profit from 107 : $20.80
profit from 108 : $19.75
profit from 109 : $19.61
profit from 110 : $18.80
 maximum profit : $22.54 at 99


From this simulation, we see that the maximum profit occurs when we buy 97 newspapers.

# 2. Mean Time Between Failures

In [6]:
"""
    mtbf(means::Vector{Float64}, deviations::Vector{Float64}, N::Int,
         verbose::Bool=false)

returns the expected lifespan and the standard deviation 
of a multicomponent product, given the means and standard deviations
of its components in the two vectors on input,
using N trials in the simulation.
If verbose, then each sample is written.
"""
function mtbf(means::Vector{Float64}, deviations::Vector{Float64}, N::Int,
             verbose::Bool=false)
   (m1, m2) = (0, 0)                       # first and second moment
   sample = [0.0 for j=1:length(means)]    # work space for each sample
   for i=1:N                               # run N simulations
       for j=1:length(means)               # randn() has mean 0, sigma 1
           sample[j] = means[j] + deviations[j]*randn()
       end
       if verbose                          # for debugging purposes ...
           println("sample : ", sample)
       end
       lifespan = minimum(sample)
       m1 = m1 + lifespan/N                # update the average
       m2 = m2 + lifespan*lifespan/N       # update second moment
   end
   return (m1, sqrt(m2 - m1*m1))           # mean and deviation
end

mtbf

We have three components:

1. The means are 11, 12, and 13.

2. The standard deviations are 1, 2, and 3.

Let us take 10,000 samples in a simulation.

In [7]:
(mu, sigma) = mtbf([11.0, 12.0, 13.0], [1.0, 2.0, 3.0], 10000)
println("expected life span : ", mu)
println("standard deviation : ", sigma)

expected life span : 10.15
standard deviation : 1.35


For a product with three components, where failure happens as soon as one of the components fails, the expected life span is 10.11, with a standard deviation of 1.37,
given that the average lifespans of the three components is 11, 12, 13.

# 3. Servicing Requests

For this problem, we need the Poisson distribution and are using the ``StatsKit``.

In [8]:
"""
    process_requests(arrivals::Vector{Float64}, jobs::Vector{Int},
                     speed::Float64, verbose::Bool=true)

returns the waiting times for the jobs with given arrival times
and sizes.  If verbose, then the algorithm will print the simulation.
"""
function process_requests(arrivals::Vector{Float64}, jobs::Vector{Int},
                          speed::Float64, verbose::Bool=true)
   wait = [0.0 for i=1:length(arrivals)]
   busy = jobs[1]*speed
   for i=2:length(arrivals)
       elapsed = arrivals[i] - arrivals[i-1]
       if elapsed >= busy
           busy = 0
       else
           busy = busy - elapsed
       end
       wait[i] = busy
       busy = busy + jobs[i]*speed
   end
   return wait
end

process_requests

The setup of the simulation is defined in the following function.

In [9]:
"""
    simulate(lambda::Int, maxsizejob::Int, speed::Float64, N::Int,
             verbose::Bool=true)

returns the average waiting time and its standard deviation,
given the lambda parameter in the Poisson distribution,
the maximum size of a job, the speed of the processor,
and the number N of time steps.
"""
function simulate(lambda::Int, maxsizejob::Int, speed::Float64,
                  N::Int, verbose::Bool=true)
    p = Poisson(lambda)
    arrivalslist = []
    jobslist = []
    elapsed = 0.0
    for i=1:N
        nbr = rand(p, 1)[1]
        if verbose
            println("at step ", i, " : ", nbr, " jobs")
        end
        dt = 1.0/nbr
        for j=1:nbr
            push!(arrivalslist, elapsed)
            size = rand(1:maxsizejob, 1)
            push!(jobslist, size[1])
            elapsed = elapsed + dt
        end
    end
    if verbose
        println("total number of arrivals : ", length(arrivalslist))
        println("total number of jobs : ", length(jobslist))
    end
    arrivals = Vector{Float64}(arrivalslist)
    jobs = Vector{Int}(jobslist)
    if verbose
        println("The arrivals : ", arrivals)
        println("The jobs : ", jobs)
    end
    w = process_requests(arrivals, jobs, speed, verbose)
    if verbose
        println("The wait times : ", w)
    end
    return mean(w), std(w)
end

simulate

Then we run the simulation for some specific parameters.

In [10]:
mu, sigma = simulate(10, 7, 0.02, 1000, false)
println("Lambda : 10, max#items : 7, speed : 0.02.");
println("Simulation with 1000 trials :")
println(" the average wait time : ", mu)
println("the standard deviation : ", sigma)

Lambda : 10, max#items : 7, speed : 0.02.
Simulation with 1000 trials :
 the average wait time : 0.17
the standard deviation : 0.27
